In [ ]:
# Import packages, load the study configuration and the tables for the CHIP analysis
from pathlib import Path

from SSc_Somatic import configure, load_config, load_study
from SSc_Somatic.chip import analysis as chip_funcs
from SSc_Somatic.loading import load_blood_sample_ids

# Study configuration. Point this at another configuration file to analyze a different cohort.
config = configure(load_config(Path('..') / 'config' / 'example_SSc.yaml'))
data = load_study(config, 'chip')

# Cohort labels and column names used below
status_col = config.require('cohort', 'status_column')
group_col = config.require('cohort', 'group_column')
meta_id_col = config.require('cohort', 'metadata_sample_column')
age_col = config.require('cohort', 'clinical_columns', 'age')
duration_col = config.require('cohort', 'clinical_columns', 'disease_duration')
case_label = config.require('cohort', 'case_label')
control_label = config.require('cohort', 'control_label')
max_age = config.require('chip', 'max_age')
bins, labels = config.age_bins, config.age_labels

# Tables loaded for the CHIP analysis
meta_df = data.metadata
snv_df = data.snv_table
processed_df = data.processed_metadata
wl_final_anno_df = data.whitelist_calls
wl_final_merged_df = data.whitelist_merged
sample_list_input = data.sample_ids

In [ ]:
# Choose whether to rebuild the CHIP tables, and set the output file locations
REGENERATE = True  # Set to False to reuse tables written by an earlier run

wl_output_filename = config.output_path('chip', 'whitelist_tidy_table')
merged_output_filename = config.output_path('chip', 'whitelist_merged_table')
processed_output_filename = config.output_path('chip', 'processed_metadata')
combined_calls_filename = config.output_path('chip', 'combined_whitelist_calls')

In [ ]:
# Generate processed_df if it doesn't exist or regenerating
if processed_df is None or REGENERATE:
    processed_df = chip_funcs.process_snv_data(snv_df, meta_df, bins, labels)
    processed_df = processed_df.loc[processed_df[status_col].isin([case_label, control_label])]
    processed_df.to_csv(processed_output_filename, index=False)
    print(f"Processed metadata table (cases and controls, with age groups): {len(processed_df)} rows")
    print(f"Saved to: {processed_output_filename}")
else:
    print(f"Loaded processed metadata table: {len(processed_df)} rows")

In [ ]:
# Keep only participants sequenced from blood
blood_ids = load_blood_sample_ids(config)
meta_df = meta_df.loc[meta_df[meta_id_col].isin(blood_ids)]
print("Metadata after keeping participants sequenced from blood (rows, columns):", meta_df.shape)

In [ ]:
# Build the annotated CHIP whitelist table: combine the per-sample whitelist files, filter
# on read depth, allele depth and strand bias, remove duplicate calls and add participant
# metadata. The table from an earlier run is reused when REGENERATE is False.
if wl_final_anno_df is None or REGENERATE:
    print("Combining the per-sample CHIP whitelist files:")
    wl_df, _ = chip_funcs.concatenate_wl_files(
        vcf_root=str(config.path('chip', 'whitelist_calls_dir')),
        sample_ids=sample_list_input,
        output_filename=combined_calls_filename.name,
        output_dir=str(combined_calls_filename.parent),
        file_pattern=config.require('chip', 'whitelist_file_pattern')
    )

    print("Filtering whitelist calls on read depth:")
    filtered_wl_df = chip_funcs.filter_variants_by_dp(wl_df)

    print("Filtering whitelist calls on allele depth and strand bias:")
    df_final1 = chip_funcs.filter_variants_by_ad_sb(filtered_wl_df)

    rows_before_dedup = len(df_final1)
    df_final = df_final1.drop_duplicates(subset=['sample_id', 'Chr', 'Start', 'Gene.refGene', 'VAF'])
    rows_after_dedup = len(df_final)
    print(f"Dropped {rows_before_dedup - rows_after_dedup} duplicate variants ({rows_before_dedup} -> {rows_after_dedup})")

    print("Adding participant metadata and age groups to the CHIP calls:")
    wl_final_anno_df, all_chip_ids, big_chip_ids, excluded_count = chip_funcs.process_wl_chip_data_complete_ages(
        df_final=df_final,
        meta_df=meta_df,
        bins=bins,
        labels=labels
    )
    wl_final_anno_df['age_at_diagnosis'] = wl_final_anno_df[age_col] - wl_final_anno_df[duration_col]
    print(f"Annotated CHIP whitelist table: {len(wl_final_anno_df)} rows")
else:
    print(f"Loaded annotated CHIP whitelist table: {len(wl_final_anno_df)} rows")

In [ ]:
# Remove CHIP samples without a subtype annotation
chip_samples_without_subtype = wl_final_anno_df.loc[
    wl_final_anno_df[group_col].isna(), 'sample_id'
].astype(str).unique().tolist()
if chip_samples_without_subtype:
    print(f"Removing {len(chip_samples_without_subtype)} samples without subtype: {chip_samples_without_subtype}")
    wl_final_anno_df = wl_final_anno_df.loc[~wl_final_anno_df['sample_id'].isin(chip_samples_without_subtype)]
    print(f"Annotated CHIP whitelist table after removal: {len(wl_final_anno_df)} rows")

# Run CHIP count analysis
print("Comparing the number of CHIP calls per sample between disease groups:")
chip_count_results = chip_funcs.comprehensive_chip_count_sample_analysis(wl_final_anno_df)
chip_count_pvalue_summary = chip_funcs.create_pvalue_summary(chip_count_results)
chip_count_significant_summary = chip_funcs.summarize_significant_findings(chip_count_results)
print("Generated CHIP count analysis results")

# Save wl_final_anno_df
wl_ids = wl_final_anno_df['sample_id'].tolist()
wl_final_anno_df.to_csv(wl_output_filename, sep='\t', index=False)
print(f"Saved the annotated CHIP whitelist table to: {wl_output_filename}")

In [ ]:
# Build the merged table (one row per participant, with SNV counts, CHIP status and
# metadata), or reuse the table written by an earlier run
if wl_final_merged_df is None or REGENERATE:
    print("Merging SNV counts, CHIP calls and metadata into one row per participant:")
    merged_df = chip_funcs.merge_snv_with_chip_and_metadata(snv_df, wl_final_anno_df, meta_df)

    wl_ids = wl_final_anno_df['sample_id'].tolist()
    merged_df['wl_CHIP'] = merged_df['sample_id'].apply(lambda x: 'WL' if x in wl_ids else 'NO_CHIP')
    merged_df['is_wl'] = (merged_df['wl_CHIP'] == 'WL').astype(int)

    merged_df.to_csv(merged_output_filename, sep='\t', index=False)
    print(f"Merged table: {len(merged_df)} rows")
    print(f"Saved to: {merged_output_filename}")
else:
    merged_df = wl_final_merged_df
    wl_ids = wl_final_anno_df['sample_id'].tolist()
    print(f"Loaded merged table: {len(merged_df)} rows")

# SNV table annotated with metadata and age groups
print("Annotating the SNV count table with metadata and age groups:")
snv_anno_ssc_df = chip_funcs.create_snv_anno_ssc_dataframe(
    snv_df=snv_df,
    meta_df=meta_df,
    age_bins=bins,
    age_labels=labels
)
print(f"Annotated SNV count table: {len(snv_anno_ssc_df)} rows")

In [ ]:
# Create age-filtered version (too few samples at or above max_age)
merged_df_age = merged_df.loc[merged_df[age_col] < max_age].copy()
merged_df_age['wl_CHIP'] = merged_df_age['sample_id'].apply(lambda x: 'WL' if x in wl_ids else 'NO_CHIP')
merged_df_age['is_wl'] = (merged_df_age['wl_CHIP'] == 'WL').astype(int)
print(f"Merged table restricted to {age_col} below {max_age}: {len(merged_df_age)} rows")